# COMPX525-26A Assignment 1: Petri Dish Cell Culture Classification

**Three Approaches:**
1. Custom CNN trained from scratch
2. Fine-tuning a pretrained EfficientNet-B0
3. Pretrained embeddings + Classical ML (SVM, XGBoost)

**Final:** Ensemble of all three for maximum accuracy

## 0. Setup & Configuration

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install extra dependencies
!pip install -q albumentations scikit-learn xgboost

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
import torchvision.transforms as T
import torchvision.models as models

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, f1_score
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

import cv2
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════
# Upload your deep_learning_1 folder to Google Drive, then set the path below.
# Example: if your folder is at My Drive > deep_learning_1, use:
#   BASE_DIR = '/content/drive/MyDrive/deep_learning_1'
# If it's inside a subfolder like My Drive > assignments > deep_learning_1:
#   BASE_DIR = '/content/drive/MyDrive/assignments/deep_learning_1'
# ══════════════════════════════════════════════════════════════════════

import os

BASE_DIR = '/content/drive/MyDrive/deep_learning_1'  # <-- UPDATE THIS

TRAIN_DIR = os.path.join(BASE_DIR, 'processed', 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'processed', 'test')

IMG_SIZE     = 224
NUM_CLASSES  = 16   # a-i, k-q (j has no data)
BATCH_SIZE   = 32
NUM_WORKERS  = 2
VAL_RATIO    = 0.15

# Verify paths
assert os.path.isdir(TRAIN_DIR), f'Train dir not found: {TRAIN_DIR}'
assert os.path.isdir(TEST_DIR),  f'Test dir not found: {TEST_DIR}'
print(f'Base dir:  {BASE_DIR}')
print(f'Train dir: {TRAIN_DIR}')
print(f'Test dir:  {TEST_DIR}')

## 1. Data Exploration & Analysis

In [ ]:
# Build class mapping (skip 'j' which has no training data)
all_classes = sorted([d for d in os.listdir(TRAIN_DIR)
                      if os.path.isdir(os.path.join(TRAIN_DIR, d))])
print(f'Classes found: {all_classes}')
print(f'Number of classes: {len(all_classes)}')

class_to_idx = {c: i for i, c in enumerate(all_classes)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
print(f'\nClass mapping: {class_to_idx}')

In [ ]:
# Count images per class
class_counts = {}
for cls in all_classes:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[cls] = len(files)

print('Class distribution (processed/augmented):')
for cls, cnt in sorted(class_counts.items()):
    bar = '█' * (cnt // 20)
    print(f'  {cls}: {cnt:5d} {bar}')
print(f'\nTotal training images: {sum(class_counts.values())}')
print(f'Test images: {len(os.listdir(TEST_DIR))}')

In [ ]:
# Visualize class distribution
fig, ax = plt.subplots(figsize=(12, 5))
classes = list(class_counts.keys())
counts = list(class_counts.values())
colors = plt.cm.tab20(np.linspace(0, 1, len(classes)))
ax.bar(classes, counts, color=colors)
ax.set_xlabel('Class')
ax.set_ylabel('Number of Images')
ax.set_title('Training Set Class Distribution (After Preprocessing & Augmentation)')
for i, (cls, cnt) in enumerate(zip(classes, counts)):
    ax.text(i, cnt + 10, str(cnt), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'class_distribution.png'), dpi=150)
plt.show()

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, cls in enumerate(all_classes):
    ax = axes[i // 8, i % 8]
    cls_dir = os.path.join(TRAIN_DIR, cls)
    # Pick a non-augmented image if possible
    files = [f for f in os.listdir(cls_dir) if not f.startswith('aug_')]
    if not files:
        files = os.listdir(cls_dir)
    img_path = os.path.join(cls_dir, files[0])
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'Class: {cls}', fontsize=10)
    ax.axis('off')
plt.suptitle('Sample Images per Class', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'sample_images.png'), dpi=150)
plt.show()

## 2. Dataset & DataLoaders

In [ ]:
class PetriDishDataset(Dataset):
    """Custom Dataset for petri dish cell culture images."""

    def __init__(self, file_list, labels, transform=None):
        self.file_list = file_list
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = self.file_list[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label


class TestDataset(Dataset):
    """Dataset for test images (no labels)."""

    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.files = sorted([f for f in os.listdir(test_dir)
                             if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        img_path = os.path.join(self.test_dir, fname)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, fname

In [ ]:
# Build file lists and labels
all_files = []
all_labels = []
for cls in all_classes:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_files.append(os.path.join(cls_dir, fname))
            all_labels.append(class_to_idx[cls])

all_files = np.array(all_files)
all_labels = np.array(all_labels)
print(f'Total samples: {len(all_files)}')

# Stratified train/val split
train_idx, val_idx = train_test_split(
    np.arange(len(all_files)), test_size=VAL_RATIO,
    stratify=all_labels, random_state=SEED
)
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}')

In [ ]:
# ── Transforms ──
# ImageNet normalization stats
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=30),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    T.RandomErasing(p=0.1, scale=(0.02, 0.1)),
])

val_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# TTA (Test-Time Augmentation) transforms
tta_transforms = [
    val_transform,  # original
    T.Compose([T.RandomHorizontalFlip(p=1.0), T.Resize((IMG_SIZE, IMG_SIZE)),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.RandomVerticalFlip(p=1.0), T.Resize((IMG_SIZE, IMG_SIZE)),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.RandomRotation(degrees=(90, 90)), T.Resize((IMG_SIZE, IMG_SIZE)),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
    T.Compose([T.RandomRotation(degrees=(180, 180)), T.Resize((IMG_SIZE, IMG_SIZE)),
               T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)]),
]

In [ ]:
# Create datasets
train_dataset = PetriDishDataset(all_files[train_idx], all_labels[train_idx], train_transform)
val_dataset   = PetriDishDataset(all_files[val_idx],   all_labels[val_idx],   val_transform)
test_dataset  = TestDataset(TEST_DIR, val_transform)

# Weighted sampler to handle class imbalance
train_label_counts = Counter(all_labels[train_idx])
weights_per_class = {cls: 1.0 / cnt for cls, cnt in train_label_counts.items()}
sample_weights = np.array([weights_per_class[label] for label in all_labels[train_idx]])
sample_weights = torch.from_numpy(sample_weights).double()
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

In [ ]:
# Compute class weights for loss function
total_samples = sum(train_label_counts.values())
class_weights = torch.tensor(
    [total_samples / (NUM_CLASSES * train_label_counts[i]) for i in range(NUM_CLASSES)],
    dtype=torch.float32
).to(DEVICE)
print('Class weights for loss:')
for i, w in enumerate(class_weights):
    print(f'  {idx_to_class[i]}: {w:.3f}')

## 3. Training Utilities

In [ ]:
class EarlyStopping:
    """Early stops training if validation loss doesn't improve."""
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


def mixup_data(x, y, alpha=0.2):
    """Applies mixup augmentation."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def train_one_epoch(model, loader, criterion, optimizer, scheduler=None, use_mixup=False):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        if use_mixup and random.random() < 0.5:
            images, targets_a, targets_b, lam = mixup_data(images, labels)
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    if scheduler is not None:
        scheduler.step()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels)


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs, model_name, use_mixup=False, patience=12):
    """Full training loop with early stopping and best model saving."""
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    early_stopping = EarlyStopping(patience=patience)
    save_path = os.path.join(BASE_DIR, f'best_{model_name}.pth')

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, use_mixup
        )
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch [{epoch+1}/{num_epochs}] '
              f'Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}% | LR: {lr:.6f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f'  >> Saved best model (val_acc={val_acc:.2f}%)')

        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f'Early stopping at epoch {epoch+1}')
            break

    # Load best model
    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'\nBest val accuracy: {best_val_acc:.2f}%')
    return model, history

In [ ]:
def plot_training_history(history, title):
    """Plot training curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history['train_loss'], label='Train Loss')
    ax1.plot(history['val_loss'], label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title(f'{title} - Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['train_acc'], label='Train Acc')
    ax2.plot(history['val_acc'], label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title(f'{title} - Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, f'{title.lower().replace(" ", "_")}_curves.png'), dpi=150)
    plt.show()


def plot_confusion_matrix(y_true, y_pred, title):
    """Plot confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=all_classes, yticklabels=all_classes, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix - {title}')
    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, f'{title.lower().replace(" ", "_")}_cm.png'), dpi=150)
    plt.show()

---
## 4. Approach 1: Custom CNN from Scratch

A CNN with residual connections, batch normalization, and dropout.
Inspired by ResNet but designed from scratch specifically for this task.

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with two conv layers and optional downsampling."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class CustomCNN(nn.Module):
    """Custom CNN with residual blocks for petri dish classification."""
    def __init__(self, num_classes=16):
        super().__init__()

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        # Residual stages
        self.layer1 = self._make_layer(64,  64,  num_blocks=2, stride=1)
        self.layer2 = self._make_layer(64,  128, num_blocks=2, stride=2)
        self.layer3 = self._make_layer(128, 256, num_blocks=2, stride=2)
        self.layer4 = self._make_layer(256, 512, num_blocks=2, stride=2)

        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(512, num_classes)

        # Weight initialization
        self._initialize_weights()

    def _make_layer(self, in_channels, out_channels, num_blocks, stride):
        layers = [ResidualBlock(in_channels, out_channels, stride)]
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels, 1))
        return nn.Sequential(*layers)

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


# Create model
custom_cnn = CustomCNN(NUM_CLASSES).to(DEVICE)
total_params = sum(p.numel() for p in custom_cnn.parameters())
trainable_params = sum(p.numel() for p in custom_cnn.parameters() if p.requires_grad)
print(f'Custom CNN - Total params: {total_params:,}, Trainable: {trainable_params:,}')

In [ ]:
# ── Train Custom CNN ──
EPOCHS_CUSTOM = 50

criterion_custom = nn.CrossEntropyLoss(weight=class_weights)
optimizer_custom = optim.AdamW(custom_cnn.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_custom = optim.lr_scheduler.CosineAnnealingLR(optimizer_custom, T_max=EPOCHS_CUSTOM, eta_min=1e-6)

print('Training Custom CNN from scratch...')
print('=' * 70)
custom_cnn, history_custom = train_model(
    custom_cnn, train_loader, val_loader, criterion_custom,
    optimizer_custom, scheduler_custom, EPOCHS_CUSTOM,
    model_name='custom_cnn', use_mixup=True, patience=15
)

In [ ]:
# Evaluate Custom CNN
plot_training_history(history_custom, 'Custom CNN')

_, val_acc_custom, preds_custom, labels_custom = evaluate(
    custom_cnn, val_loader, criterion_custom
)
print(f'\nCustom CNN Validation Accuracy: {val_acc_custom:.2f}%')
print('\nClassification Report:')
print(classification_report(labels_custom, preds_custom, target_names=all_classes))
plot_confusion_matrix(labels_custom, preds_custom, 'Custom CNN')

---
## 5. Approach 2: Fine-tuning Pretrained EfficientNet-B0

Transfer learning using EfficientNet-B0 pretrained on ImageNet.
Progressive unfreezing strategy: first train only the classifier head, then gradually unfreeze deeper layers.

In [ ]:
def create_efficientnet(num_classes, freeze_backbone=True):
    """Create EfficientNet-B0 with a custom classifier head."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.features.parameters():
            param.requires_grad = False

    # Replace classifier
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, num_classes)
    )
    return model


efficientnet = create_efficientnet(NUM_CLASSES, freeze_backbone=True).to(DEVICE)
trainable = sum(p.numel() for p in efficientnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in efficientnet.parameters())
print(f'EfficientNet-B0 - Total: {total:,}, Trainable (head only): {trainable:,}')

In [ ]:
# ── Phase 1: Train only the classifier head ──
EPOCHS_HEAD = 10

criterion_eff = nn.CrossEntropyLoss(weight=class_weights)
optimizer_eff = optim.Adam(efficientnet.classifier.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler_eff = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff, T_max=EPOCHS_HEAD, eta_min=1e-5)

print('Phase 1: Training classifier head only...')
print('=' * 70)
efficientnet, history_eff_phase1 = train_model(
    efficientnet, train_loader, val_loader, criterion_eff,
    optimizer_eff, scheduler_eff, EPOCHS_HEAD,
    model_name='efficientnet_phase1', patience=10
)

In [ ]:
# ── Phase 2: Unfreeze all layers, fine-tune with lower LR ──
EPOCHS_FINETUNE = 30

for param in efficientnet.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in efficientnet.parameters() if p.requires_grad)
print(f'Trainable parameters (full model): {trainable:,}')

# Use discriminative learning rates: lower LR for pretrained layers
param_groups = [
    {'params': efficientnet.features.parameters(), 'lr': 1e-5},
    {'params': efficientnet.classifier.parameters(), 'lr': 5e-4},
]
optimizer_eff2 = optim.AdamW(param_groups, weight_decay=1e-4)
scheduler_eff2 = optim.lr_scheduler.CosineAnnealingLR(optimizer_eff2, T_max=EPOCHS_FINETUNE, eta_min=1e-7)

print('\nPhase 2: Fine-tuning full model...')
print('=' * 70)
efficientnet, history_eff_phase2 = train_model(
    efficientnet, train_loader, val_loader, criterion_eff,
    optimizer_eff2, scheduler_eff2, EPOCHS_FINETUNE,
    model_name='efficientnet_finetuned', use_mixup=True, patience=12
)

In [ ]:
# Combine histories for plotting
history_eff = {k: history_eff_phase1[k] + history_eff_phase2[k] for k in history_eff_phase1}

plot_training_history(history_eff, 'EfficientNet-B0 Fine-tuned')

_, val_acc_eff, preds_eff, labels_eff = evaluate(efficientnet, val_loader, criterion_eff)
print(f'\nEfficientNet-B0 Validation Accuracy: {val_acc_eff:.2f}%')
print('\nClassification Report:')
print(classification_report(labels_eff, preds_eff, target_names=all_classes))
plot_confusion_matrix(labels_eff, preds_eff, 'EfficientNet-B0')

---
## 6. Approach 3: Pretrained Embeddings + Classical ML

Extract deep features from a pretrained ResNet-50, then train SVM, Random Forest, and XGBoost classifiers.

In [ ]:
@torch.no_grad()
def extract_features(model, loader, is_test=False):
    """Extract feature embeddings from the model's penultimate layer."""
    model.eval()
    features_list = []
    labels_list = []

    for batch in loader:
        images = batch[0].to(DEVICE)
        feats = model(images)
        features_list.append(feats.cpu().numpy())
        if not is_test:
            labels_list.append(batch[1].numpy())

    features = np.concatenate(features_list, axis=0)
    if not is_test:
        labels = np.concatenate(labels_list, axis=0)
        return features, labels
    return features


# Load pretrained ResNet-50 as feature extractor
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
# Remove the final FC layer -> get 2048-dim features
feature_extractor = nn.Sequential(*list(resnet50.children())[:-1], nn.Flatten()).to(DEVICE)
feature_extractor.eval()

print('Extracting training features...')
# Use val_transform for consistent feature extraction
embed_train_dataset = PetriDishDataset(all_files[train_idx], all_labels[train_idx], val_transform)
embed_val_dataset   = PetriDishDataset(all_files[val_idx],   all_labels[val_idx],   val_transform)
embed_train_loader = DataLoader(embed_train_dataset, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
embed_val_loader   = DataLoader(embed_val_dataset,   batch_size=64, shuffle=False, num_workers=NUM_WORKERS)

X_train_feat, y_train_feat = extract_features(feature_extractor, embed_train_loader)
X_val_feat, y_val_feat     = extract_features(feature_extractor, embed_val_loader)

print(f'Train features shape: {X_train_feat.shape}')
print(f'Val features shape:   {X_val_feat.shape}')

# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_val_scaled   = scaler.transform(X_val_feat)

In [ ]:
# ── SVM ──
print('Training SVM (RBF kernel)...')
svm_clf = SVC(kernel='rbf', C=10.0, gamma='scale', class_weight='balanced',
              decision_function_shape='ovr', probability=True, random_state=SEED)
svm_clf.fit(X_train_scaled, y_train_feat)
svm_preds = svm_clf.predict(X_val_scaled)
svm_acc = accuracy_score(y_val_feat, svm_preds) * 100
print(f'SVM Validation Accuracy: {svm_acc:.2f}%')
print(classification_report(y_val_feat, svm_preds, target_names=all_classes))

In [ ]:
# ── XGBoost ──
print('Training XGBoost...')
xgb_clf = XGBClassifier(
    n_estimators=500, max_depth=8, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    use_label_encoder=False, eval_metric='mlogloss',
    random_state=SEED, n_jobs=-1,
    tree_method='hist'
)
xgb_clf.fit(X_train_scaled, y_train_feat,
            eval_set=[(X_val_scaled, y_val_feat)],
            verbose=50)
xgb_preds = xgb_clf.predict(X_val_scaled)
xgb_acc = accuracy_score(y_val_feat, xgb_preds) * 100
print(f'XGBoost Validation Accuracy: {xgb_acc:.2f}%')
print(classification_report(y_val_feat, xgb_preds, target_names=all_classes))

In [ ]:
# ── Random Forest ──
print('Training Random Forest...')
rf_clf = RandomForestClassifier(
    n_estimators=500, max_depth=None, class_weight='balanced',
    random_state=SEED, n_jobs=-1
)
rf_clf.fit(X_train_scaled, y_train_feat)
rf_preds = rf_clf.predict(X_val_scaled)
rf_acc = accuracy_score(y_val_feat, rf_preds) * 100
print(f'Random Forest Validation Accuracy: {rf_acc:.2f}%')
print(classification_report(y_val_feat, rf_preds, target_names=all_classes))

In [ ]:
# Compare Approach 3 methods
print('\n' + '='*50)
print('Approach 3 Summary (Embeddings + ML):')
print(f'  SVM:           {svm_acc:.2f}%')
print(f'  XGBoost:       {xgb_acc:.2f}%')
print(f'  Random Forest: {rf_acc:.2f}%')

# Best ML classifier for ensemble
ml_results = {'svm': (svm_acc, svm_clf), 'xgb': (xgb_acc, xgb_clf), 'rf': (rf_acc, rf_clf)}
best_ml_name = max(ml_results, key=lambda k: ml_results[k][0])
best_ml_acc, best_ml_clf = ml_results[best_ml_name]
print(f'\nBest ML classifier: {best_ml_name} ({best_ml_acc:.2f}%)')

# Plot confusion matrix for best ML
best_ml_preds = best_ml_clf.predict(X_val_scaled)
plot_confusion_matrix(y_val_feat, best_ml_preds, f'Embeddings + {best_ml_name.upper()}')

---
## 7. Ensemble & Final Predictions

Combine predictions from all three approaches using soft voting (probability averaging).

In [ ]:
@torch.no_grad()
def get_probabilities(model, loader):
    """Get softmax probabilities from a PyTorch model."""
    model.eval()
    all_probs = []
    for images, _ in loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        all_probs.append(probs.cpu().numpy())
    return np.concatenate(all_probs, axis=0)


@torch.no_grad()
def get_test_probabilities(model, test_dir, transforms_list):
    """Get TTA probabilities for test set."""
    model.eval()
    all_probs = []

    for t in transforms_list:
        ds = TestDataset(test_dir, t)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
        probs = []
        for images, _ in loader:
            images = images.to(DEVICE)
            outputs = model(images)
            probs.append(F.softmax(outputs, dim=1).cpu().numpy())
        all_probs.append(np.concatenate(probs, axis=0))

    # Average over TTA transforms
    return np.mean(all_probs, axis=0)


# ── Validation ensemble ──
print('Computing validation ensemble...')
val_probs_custom = get_probabilities(custom_cnn, val_loader)
val_probs_eff    = get_probabilities(efficientnet, val_loader)

# ML probabilities
if hasattr(best_ml_clf, 'predict_proba'):
    val_probs_ml = best_ml_clf.predict_proba(X_val_scaled)
else:
    # One-hot encode predictions as fallback
    val_probs_ml = np.eye(NUM_CLASSES)[best_ml_clf.predict(X_val_scaled)]

# Weighted ensemble (EfficientNet gets higher weight as fine-tuned model)
w_custom, w_eff, w_ml = 0.2, 0.5, 0.3
val_probs_ensemble = w_custom * val_probs_custom + w_eff * val_probs_eff + w_ml * val_probs_ml
val_preds_ensemble = val_probs_ensemble.argmax(axis=1)

ensemble_acc = accuracy_score(all_labels[val_idx], val_preds_ensemble) * 100
print(f'\nEnsemble Validation Accuracy: {ensemble_acc:.2f}%')
print(f'\n--- Comparison ---')
print(f'Custom CNN:     {val_acc_custom:.2f}%')
print(f'EfficientNet:   {val_acc_eff:.2f}%')
print(f'Best ML:        {best_ml_acc:.2f}%')
print(f'Ensemble:       {ensemble_acc:.2f}%')

print('\nEnsemble Classification Report:')
print(classification_report(all_labels[val_idx], val_preds_ensemble, target_names=all_classes))
plot_confusion_matrix(all_labels[val_idx], val_preds_ensemble, 'Ensemble')

In [ ]:
# ── Generate Test Predictions with TTA ──
print('Generating test predictions with TTA...')

# CNN models with TTA
test_probs_custom = get_test_probabilities(custom_cnn, TEST_DIR, tta_transforms)
test_probs_eff    = get_test_probabilities(efficientnet, TEST_DIR, tta_transforms)

# ML model - extract test features
embed_test_loader = DataLoader(
    TestDataset(TEST_DIR, val_transform),
    batch_size=64, shuffle=False, num_workers=NUM_WORKERS
)
test_features = extract_features(feature_extractor, embed_test_loader, is_test=True)
test_features_scaled = scaler.transform(test_features)

if hasattr(best_ml_clf, 'predict_proba'):
    test_probs_ml = best_ml_clf.predict_proba(test_features_scaled)
else:
    test_probs_ml = np.eye(NUM_CLASSES)[best_ml_clf.predict(test_features_scaled)]

# Weighted ensemble
test_probs_ensemble = w_custom * test_probs_custom + w_eff * test_probs_eff + w_ml * test_probs_ml
test_preds = test_probs_ensemble.argmax(axis=1)

# Get test filenames
test_files = sorted([f for f in os.listdir(TEST_DIR)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

# Map predictions back to class names
test_classes = [idx_to_class[p] for p in test_preds]

print(f'Test predictions generated for {len(test_files)} images')
print(f'Predicted class distribution: {Counter(test_classes)}')

In [ ]:
# ── Save CSV Submission ──
submission = pd.DataFrame({
    'TestFileName': test_files,
    'Class': test_classes
})

csv_path = os.path.join(BASE_DIR, 'submission.csv')

# Write with exact format from README: 'TestFileName, Class' (space after comma)
with open(csv_path, 'w') as f:
    f.write('TestFileName, Class\n')
    for _, row in submission.iterrows():
        f.write(f"{row['TestFileName']}, {row['Class']}\n")

print(f'Submission saved to: {csv_path}')
print(f'\nFirst 10 rows:')
print(submission.head(10).to_string(index=False))
print(f'\nTotal predictions: {len(submission)}')
print(f'\nPredicted class distribution:')
print(submission['Class'].value_counts().sort_index())

In [ ]:
# ── Auto-save all results to JSON for report generation ──
import json as _json

results = {
    'custom_cnn': {
        'val_accuracy': round(val_acc_custom, 2),
        'history': {k: [round(v, 4) for v in vals] for k, vals in history_custom.items()},
        'classification_report': classification_report(labels_custom, preds_custom, target_names=all_classes, output_dict=True),
    },
    'efficientnet': {
        'val_accuracy': round(val_acc_eff, 2),
        'history': {k: [round(v, 4) for v in vals] for k, vals in history_eff.items()},
        'classification_report': classification_report(labels_eff, preds_eff, target_names=all_classes, output_dict=True),
    },
    'embeddings_ml': {
        'svm_accuracy': round(svm_acc, 2),
        'xgboost_accuracy': round(xgb_acc, 2),
        'random_forest_accuracy': round(rf_acc, 2),
        'best_ml': best_ml_name,
        'best_ml_accuracy': round(best_ml_acc, 2),
        'classification_report': classification_report(y_val_feat, best_ml_preds, target_names=all_classes, output_dict=True),
    },
    'ensemble': {
        'val_accuracy': round(ensemble_acc, 2),
        'weights': {'custom': w_custom, 'efficientnet': w_eff, 'ml': w_ml},
        'classification_report': classification_report(all_labels[val_idx], val_preds_ensemble, target_names=all_classes, output_dict=True),
    },
    'test_predictions': {
        'total': len(test_files),
        'class_distribution': dict(Counter(test_classes)),
    },
    'dataset_info': {
        'num_classes': NUM_CLASSES,
        'classes': all_classes,
        'train_size': len(train_idx),
        'val_size': len(val_idx),
        'test_size': len(test_files),
        'class_counts': class_counts,
    }
}

results_path = os.path.join(BASE_DIR, 'results.json')
with open(results_path, 'w') as f:
    _json.dump(results, f, indent=2, default=str)

print(f'Results saved to: {results_path}')
print(f'\n=== QUICK SUMMARY ===')
print(f'Custom CNN:   {results["custom_cnn"]["val_accuracy"]}%')
print(f'EfficientNet: {results["efficientnet"]["val_accuracy"]}%')
print(f'Best ML ({best_ml_name}): {results["embeddings_ml"]["best_ml_accuracy"]}%')
print(f'Ensemble:     {results["ensemble"]["val_accuracy"]}%')

---
## 8. Summary & Model Comparison

In [ ]:
# Final comparison bar chart
model_names = ['Custom CNN', 'EfficientNet-B0\n(Fine-tuned)', f'Embeddings +\n{best_ml_name.upper()}', 'Ensemble']
accuracies = [val_acc_custom, val_acc_eff, best_ml_acc, ensemble_acc]
colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(model_names, accuracies, color=colors_bar, edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
            f'{acc:.2f}%', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Validation Accuracy (%)', fontsize=12)
ax.set_title('Model Comparison - Validation Accuracy', fontsize=14)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'model_comparison.png'), dpi=150)
plt.show()

print('\n' + '='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
for name, acc in zip(model_names, accuracies):
    print(f'  {name:30s} {acc:.2f}%')
print('='*60)

In [ ]:
# Per-class accuracy comparison across models
from sklearn.metrics import precision_recall_fscore_support

def per_class_accuracy(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    per_class = cm.diagonal() / cm.sum(axis=1)
    return {c: a for c, a in zip(class_names, per_class)}

pca_custom = per_class_accuracy(labels_custom, preds_custom, all_classes)
pca_eff = per_class_accuracy(labels_eff, preds_eff, all_classes)
pca_ml = per_class_accuracy(y_val_feat, best_ml_preds, all_classes)
pca_ensemble = per_class_accuracy(all_labels[val_idx], val_preds_ensemble, all_classes)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(all_classes))
width = 0.2
ax.bar(x - 1.5*width, [pca_custom[c] for c in all_classes], width, label='Custom CNN', color='#3498db')
ax.bar(x - 0.5*width, [pca_eff[c] for c in all_classes], width, label='EfficientNet', color='#2ecc71')
ax.bar(x + 0.5*width, [pca_ml[c] for c in all_classes], width, label=f'{best_ml_name.upper()}', color='#e74c3c')
ax.bar(x + 1.5*width, [pca_ensemble[c] for c in all_classes], width, label='Ensemble', color='#f39c12')

ax.set_xticks(x)
ax.set_xticklabels(all_classes)
ax.set_xlabel('Class')
ax.set_ylabel('Accuracy')
ax.set_title('Per-Class Accuracy Comparison')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'per_class_accuracy.png'), dpi=150)
plt.show()